In [1]:
import numpy as np
import matplotlib.pyplot as plt
import polars as pl 

# Statistical Decision Theory and Optimal Stopping Applied to Credit Cards

# The Idea
I was making a presentation connecting my phd research with credit risk modeling, and one of the examples I came up with during my Aim1 was an optimal stopping problem approach to deciding if you should cancel someone's credit card or not

# The Approach
I think I need to calculate the expected loss/gain of cancelling/not cancelling someone's card. 
This might not be useful... If someone has a positive future expected value, you keep them, obviously. 
The real issue is actually correctly calculating all the probabilities.

# Anticipated Problems
1. This problem is also "infinite" time. 
   - How far out do we simulate? 
   - 

I basically want a time series of someone's expected gain over time! Then we want to stop at the point that minimizes future losses.
- Is there a weird situation that you might stop someone early? I guess you can cancel at any time, so might as stop them right when they cross over into negative expected loss.

- So this is like forecasting, but I need a dynamic equation that models EG/EL as a function of time, with modifying parameters determined by the features of the model.

# Types of Customers
1. Transactors
  - They pay off their cards in full every month, accruing no interest
  - low risk, low profits
  - Easy decision: keep them

2. Debtors
  - They have debt on their credit card that accrues interest, but they pay it off
  - Medium risk, medium profits
  - Hard decision: should we cancel now? or wait it out? 
  - When do they transition to the Defaulters category? 

3. Defaulters
  - They haven't paid off, and no signs point to them paying it off soon
  - High risk, low profits
  - Easy decision: cancel their card
  

# Approach
1. Let's take the GiveMeSomeCredit data, and calculate the expected gain/loss of having them as a customer
2. I don't know how to train these data...

# Load Data

In [6]:
df = pl.read_excel("data/default_of_credit_card_clients.xls", read_options={'header_row': 1})
rename_col_dict = {
    "PAY_0": "PAY_STATUS_SEP", 
    "PAY_2": "PAY_STATUS_AUG",
    "PAY_3": "PAY_STATUS_JUL",
    "PAY_4": "PAY_STATUS_JUN",
    "PAY_5": "PAY_STATUS_MAY",
    "PAY_6": "PAY_STATUS_APR",
    "BILL_AMT1": "BILL_AMT_SEP",
    "BILL_AMT2": "BILL_AMT_AUG",
    "BILL_AMT3": "BILL_AMT_JUL",
    "BILL_AMT4": "BILL_AMT_JUN",
    "BILL_AMT5": "BILL_AMT_MAY",
    "BILL_AMT6": "BILL_AMT_APR",
    "PAY_AMT1": "PAY_AMT_SEP",
    "PAY_AMT2": "PAY_AMT_AUG",
    "PAY_AMT3": "PAY_AMT_JUL",
    "PAY_AMT4": "PAY_AMT_JUN",
    "PAY_AMT5": "PAY_AMT_MAY",
    "PAY_AMT6": "PAY_AMT_APR",
    "default payment next month": "DEFAULT_PAYMENT_NEXT_MONTH"
}
months = [
    "SEP",
    "AUG",
    "JUL",
    "JUN",
    "MAY",
    "APR",
]
df = df.rename(rename_col_dict)
rename_col_dict.pop('default payment next month')
df.head()

ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_STATUS_SEP,PAY_STATUS_AUG,PAY_STATUS_JUL,PAY_STATUS_JUN,PAY_STATUS_MAY,PAY_STATUS_APR,BILL_AMT_SEP,BILL_AMT_AUG,BILL_AMT_JUL,BILL_AMT_JUN,BILL_AMT_MAY,BILL_AMT_APR,PAY_AMT_SEP,PAY_AMT_AUG,PAY_AMT_JUL,PAY_AMT_JUN,PAY_AMT_MAY,PAY_AMT_APR,DEFAULT_PAYMENT_NEXT_MONTH
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


# Long Format DF

In [53]:
# I want a new df with payment,bill information in long format
# Cols = [month, pay_status, bill_amt, pay_amt]
money_df = df.clone()
value_cols = list(rename_col_dict.values())
pay_cols = [v for v in value_cols if 'PAY_AMT' in v]
temp_df = df.unpivot(
    on=value_cols,
    index='ID'
)

money_df = pl.DataFrame().with_columns(
    idx = temp_df.filter(pl.col("variable").str.contains("PAY_STATUS"))['ID'],
    month = temp_df.filter(pl.col("variable").str.contains("PAY_STATUS"))['variable'].str.tail(3),
    pay_status = temp_df.filter(pl.col("variable").str.contains("PAY_STATUS"))['value'],
    bill_amt = temp_df.filter(pl.col("variable").str.contains("BILL_AMT"))['value'],
    pay_amt = temp_df.filter(pl.col("variable").str.contains("PAY_AMT"))['value'],
)
# Add debt column
money_df = money_df.with_columns(
    unpaid_amt = pl.col("bill_amt").sub(pl.col("pay_amt"))

)
money_df.filter(pl.col("idx")==8)

idx,month,pay_status,bill_amt,pay_amt,unpaid_amt
i64,str,i64,i64,i64,i64
8,"""SEP""",0,11876,380,11496
8,"""AUG""",-1,380,601,-221
8,"""JUL""",-1,601,0,601
8,"""JUN""",0,221,581,-360
8,"""MAY""",0,-159,1687,-1846
8,"""APR""",-1,567,1542,-975


## Calculate Expected Profit and Loss on each month
$EP = \alpha * bill + (unpaid + \beta * unpaid) * (1 - P(Default)) $

$  \alpha :=$ transaction fee

$  \beta := $ interest rate 

$EL = P(Default) * EAD * LGD $

$ EG = EP - EL $

- I'm simplifying for the time being. E[L] = PD * LGD * EAD, usually. I'm assuming LGD is 100% (meaning you won't recover any losses through collateral)

In [ ]:
TXN_FEE = 0.02
INT_RATE = 0.2

def expected_gain(row, prob_default):
    for month in months:
        unpaid_amt = row[f'BILL_AMT_{month}'] - row[f'PAY_AMT_{month}']
        expected_profit = TXN_FEE*row[f"BILL_AMT_{month}"] + (unpaid_amt + INT_RATE*unpaid_amt)*(1 - prob_default)
        expected_loss = prob_default * row['EAD'] * row["LGD"]
        
    return TXN_FEE*row[]

In [3]:
:w

SyntaxError: invalid syntax (459021012.py, line 1)